# Stage 2 – Match filters

**Goal:** keep only the matches that belong to the study.

A match is kept when it is a **men's international** match, **England** plays, the format is **Test, ODI or
T20I**, and the **first day** lies between 1 January 2020 and 29 October 2026. Every other file gets one written
reason for being left out.

**Reads:** the extracted match files
**Writes:** `data/interim/file_audit.csv` (every file), `data/processed/match_metadata.csv` (included matches)

In [1]:
%run ./library/lib_00_settings.ipynb
%run ./library/lib_01_data_loading.ipynb
stage_log = get_stage_logger("02_filter")

/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
file_audit = audit_match_files(list_match_files(extract_archive_if_needed()))
write_table(file_audit, INTERIM_FOLDER / "file_audit.csv")

included_matches = file_audit[file_audit["included"]].drop(columns=["included", "exclusion_reason"])
write_table(included_matches, PROCESSED_FOLDER / "match_metadata.csv")

### What was kept and why the rest was left out

In [3]:
stage_log.info("files audited: %d, included: %d", len(file_audit), len(included_matches))
stage_log.info("exclusion reasons:\n%s",
               file_audit.loc[~file_audit["included"], "exclusion_reason"].value_counts().to_string())
stage_log.info("included by format:\n%s", included_matches["format"].value_counts().reindex(list(FORMATS)).to_string())
stage_log.info("included window: %s .. %s", included_matches["start_date"].min(), included_matches["start_date"].max())

# Some limited-overs matches were abandoned before England batted. They stay in the table but give no partnerships.
matches_without_england_innings = included_matches[included_matches["n_innings_england"] == 0]
stage_log.info("included matches with no England innings: %d %s",
               len(matches_without_england_innings), matches_without_england_innings["match_id"].tolist())

2026-11-16 20:27:17,049 INFO batnet.02_filter: files audited: 961, included: 287


2026-11-16 20:27:17,050 INFO batnet.02_filter: exclusion reasons:
exclusion_reason
before_window    674


2026-11-16 20:27:17,051 INFO batnet.02_filter: included by format:
format
Test     81
ODI      88
T20I    118


2026-11-16 20:27:17,052 INFO batnet.02_filter: included window: 2020-01-03 .. 2026-10-29


2026-11-16 20:27:17,053 INFO batnet.02_filter: included matches with no England innings: 7 ['1185311', '1239536', '1276912', '1415706', '1433377', '1491719', '1496574']
